# Importation des bibliothèques

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, when, to_date

# Constantes

In [8]:
# chemin pour accéder aux données
FILE_PATH = '../src/data/donnees_immobilieres.parquet'

# Application

In [ ]:
# initialisation SparkSession
spark = SparkSession.builder.appName("RealStateProcessing").getOrCreate()

FileNotFoundError: [WinError 2] Le fichier spécifié est introuvable

In [ ]:
# chargement des données
df_raw = spark.read.parquet(FILE_PATH)

In [ ]:
# Affichage du schéma
df_raw.printSchema()

In [ ]:
# Affichage des 5 premières lignes
df_raw.show(5)

In [ ]:
# suppression des doublons suivant l'id
df = df_raw.dropduplicates('id')

In [ ]:
# filtrage pour ne garder que les biens en France
df = df.filter(col("pays") == "France")

In [ ]:
# conversion de la colonne "date_invnetaire" en type date
df = df.withColumn("date_inventaire", to_date(col("date_inventaire"), "yyyy-MM"))

In [ ]:
# mise en minuscule de la colonne "ville" pour uniformiser les données
df = df.withColumn("ville", lower(col("ville")))

In [ ]:
# mise en minuscule de la colonne "region" pour uniformiser les données
df = df.withColumn("region", lower(col("region")))

In [ ]:
# anonymisation des données sensibles
df = df.withColumn("ville",when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("ville")))

In [ ]:
# anonymisation des données sensibles
df = df.withColumn("code_postal", when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("code_postal")))

In [ ]:
# anonymisation des données sensibles
df = df.withColumn("adresse", when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("adresse")))

In [ ]:
# uniformisation des données du ministère
df = df.withColumn("ministere", when(lower(col("ministere")).like("%intérieur%"), "ministère de l'Intérieur").otherwise(lower(col("ministere"))))
.withColumn("ministere", when(lower(col("ministere")).like("%éducation%"), "ministère de l'Éducation Nationale").otherwise(col("ministere")))
.withColumn("ministere", when(lower(col("ministere")).like("%justice%"), "ministère de la Justice").otherwise(col("ministere")))
.withColumn("ministere", when(lower(col("ministere")).like("%santé%"), "ministère de la Santé").otherwise(col("ministere")))
.withColumn("ministere", when(lower(col("ministere")).like("%Ecologie%") | lower(col("ministere")).like("%environnement%"), "ministère de l'Ecologie, développement et aménagement durables'").otherwise(col("ministere")))

In [ ]:
# Affichage du schéma
df.printSchema()

In [ ]:
# Affichage des 5 premières lignes
df.show(5)